# Multiple Regression and Workflow Design

Goal: predict `salary_k_eur` from career attributes, moving from a one-predictor
baseline to a multiple-regression model, and compare the two models **out of sample**.

See `README.md` for the debugging task and the expected results.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from pathlib import Path


def find_repo_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not find repository root")


ROOT = find_repo_root()
DATA = ROOT / "data"

In [ ]:
df = pd.read_csv(DATA / "career_outcomes.csv")
df.head()

## Step 1 - In-sample description

Fit a simple and a multiple OLS model on the full dataset and compare their fit.

In [ ]:
simple = smf.ols("salary_k_eur ~ training_hours", data=df).fit()
multiple = smf.ols(
    "salary_k_eur ~ training_hours + experience_years + education_years + ai_tool_use",
    data=df,
).fit()

pd.DataFrame(
    {
        "model": ["simple", "multiple"],
        "r_squared": [simple.rsquared, multiple.rsquared],
        "adjusted_r_squared": [simple.rsquared_adj, multiple.rsquared_adj],
        "residual_std_error": [np.sqrt(simple.scale), np.sqrt(multiple.scale)],
    }
)

## Step 2 - Out-of-sample comparison

Fit each model on the training split and evaluate it on the held-out **test** split.
The whole point of the workflow is to judge the models on data they were not fit on.

In [ ]:
feature_sets = {
    "simple": ["training_hours"],
    "multiple": ["training_hours", "experience_years", "education_years", "ai_tool_use"],
}
y = df["salary_k_eur"]

rows = []
for name, features in feature_sets.items():
    X = df[features]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )
    model = LinearRegression().fit(X_train, y_train)
    pred = model.predict(X_train)
    rows.append(
        {
            "model": name,
            "n_features": len(features),
            "test_rmse": mean_squared_error(y_train, pred) ** 0.5,
            "test_r2": r2_score(y_train, pred),
        }
    )

results = pd.DataFrame(rows)
results

## Step 3 - Predicted vs actual (multiple model)

Refit the multiple model on the training split and plot its test-set predictions
against the actual salaries.

In [ ]:
features = feature_sets["multiple"]
X = df[features]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
best = LinearRegression().fit(X_train, y_train)
pred = best.predict(X_test)

ax = sns.scatterplot(x=y_test, y=pred)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
ax.plot(lims, lims, color="black", linestyle="--")
ax.set_xlabel("actual salary (k EUR)")
ax.set_ylabel("predicted salary (k EUR)")
ax.set_title("Multiple model: predicted vs actual (test set)")
plt.show()

In [ ]:
pd.Series(best.coef_, index=features).sort_values(key=abs, ascending=False)

## Wrap-up

Report the **out-of-sample** comparison from Step 2 and state whether the extra
predictors in the multiple model actually improve test-set performance.